# RetailMind AI — Day 6: CNN Image Classification using Fashion-MNIST

## Objective

Add an image-classification module to RetailMind AI using a Convolutional Neural Network (CNN).
We will use **Fashion-MNIST provided directly by Keras**. No Kaggle download, CSV file,
or separate image folder is required.

Fashion-MNIST contains 70,000 grayscale clothing images:
- 60,000 training images
- 10,000 test images
- Image size: 28 × 28 pixels
- 10 clothing categories

## Project coding approach

The notebook explains and runs the workflow. Reusable implementation is kept in `src/`:

- `data_loader.py` — load and preprocess Fashion-MNIST.
- `model.py` — build and compile the CNN.
- `train.py` — reusable training function.
- `evaluate.py` — test evaluation, confusion matrix and model saving.
- `predict.py` — predict the category of one image.

The notebook imports these functions instead of duplicating their implementation.

## 1. Import libraries and reusable functions

Run the notebook from the RetailMind project root or from its `notebooks/` folder.
The code below identifies the project root and makes `src` importable.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Add the project root to Python's import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.image_classification import (
    CLASS_NAMES,
    load_fashion_mnist,
    build_cnn,
    train_cnn,
    evaluate_cnn,
    save_model,
    load_saved_model,
    predict_image,
    save_evaluation_metrics
)

print("Imports successful!")

ModuleNotFoundError: No module named 'src.data_loader'

## 2. Load Fashion-MNIST directly from Keras

`keras.datasets.fashion_mnist.load_data()` returns two tuples:

- `(x_train, y_train)` — 60,000 training images and labels.
- `(x_test, y_test)` — 10,000 test images and labels.

Keras provides the predefined split, so we do not need to call `train_test_split`.

In [ ]:
(x_train, y_train), (x_test, y_test) = load_fashion_mnist()

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test :", x_test.shape)
print("y_test :", y_test.shape)
print("Pixel minimum:", x_train.min())
print("Pixel maximum:", x_train.max())

## 3. Understand the image dimensions and labels

The original images are grayscale and have dimensions 28 × 28. The reusable loader
normalizes pixel values from 0–255 to 0–1 and adds a channel dimension.

CNN input shape is therefore `(28, 28, 1)`, where `1` represents one grayscale channel.
Labels are integer values from 0 to 9.

In [ ]:
class_names = get_class_names()

print("Number of categories:", len(class_names))
for label, name in enumerate(class_names):
    print(label, ":", name)

print("\nOne image shape:", x_train[0].shape)
print("One label:", y_train[0], "->", class_names[y_train[0]])

## 4. Visualize sample Fashion-MNIST images

Visualizing the images helps us understand what the CNN will learn to classify.
Each title shows the category associated with the image label.

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 5. Understand the CNN architecture

A CNN learns visual patterns directly from image pixels.

- **Convolution layer (`Conv2D`):** applies learnable filters to detect patterns such as edges and textures.
- **ReLU:** adds non-linearity.
- **Max pooling:** reduces the spatial size of feature maps.
- **Flatten:** converts feature maps into a one-dimensional vector.
- **Dense:** learns combinations of extracted features.
- **Dropout:** helps reduce overfitting during training.
- **Softmax:** produces a probability for each of the 10 categories.

We use `sparse_categorical_crossentropy` because the labels are integer class IDs,
not one-hot encoded vectors.

## 6. Build the CNN using the reusable `src` function

The architecture is implemented in `src/model.py`. Here the notebook only calls the
function and inspects the returned model.

In [ ]:
model = build_cnn(
    input_shape=(28, 28, 1),
    num_classes=len(class_names),
)

model.summary()

## 7. Train the CNN

The training function is implemented in `src/train.py`.

- The model learns from `x_train` and `y_train`.
- 10% of the training data is used for validation.
- The separate 10,000-image test set remains untouched until final evaluation.
- Early stopping restores the weights from the best validation-loss epoch.
- ReduceLROnPlateau lowers the learning rate if validation loss stops improving.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

history = train_model(
    model=model,
    x_train=x_train,
    y_train=y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.10,
)

## 8. Visualize training and validation performance

Training accuracy shows performance on examples used for learning. Validation accuracy
shows performance on the held-out validation portion of the training data.

A widening gap between training and validation accuracy can be a sign of overfitting.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Accuracy across epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Loss across epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 9. Evaluate on the test dataset

The test dataset contains images that were not used to train the model or select
its best epoch. We use it for the final evaluation.

The reusable evaluation function returns test loss, test accuracy, predicted labels
and a confusion matrix.

In [ ]:
results = evaluate_model(
    model=model,
    x_test=x_test,
    y_test=y_test,
)

print(f"Test loss: {results['test_loss']:.4f}")
print(f"Test accuracy: {results['test_accuracy']:.4f}")

## 10. Confusion matrix

The confusion matrix compares actual labels with predicted labels.

- Rows represent the actual class.
- Columns represent the predicted class.
- The diagonal contains correct predictions.
- Off-diagonal values show misclassifications.

This helps identify categories the CNN commonly confuses.

In [ ]:
cm = results["confusion_matrix"]

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap="Blues")
plt.title("Fashion-MNIST Confusion Matrix")
plt.xlabel("Predicted category")
plt.ylabel("Actual category")
plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(range(len(class_names)), class_names)
plt.colorbar()

for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        plt.text(col, row, cm[row, col], ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## 11. Save the confusion matrix and test metrics

We save the confusion matrix as a CSV file and the test metrics as JSON.
These are generated artifacts, so they will appear after the notebook is run.

In [ ]:
import csv
import json

confusion_path = REPORTS_DIR / "day6_fashion_mnist_confusion_matrix.csv"
with confusion_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Actual / Predicted", *class_names])
    for label_name, row in zip(class_names, cm):
        writer.writerow([label_name, *[int(value) for value in row]])

metrics = {
    "model": "CNN - Fashion-MNIST",
    "test_loss": round(results["test_loss"], 4),
    "test_accuracy": round(results["test_accuracy"], 4),
    "train_images": int(len(x_train)),
    "test_images": int(len(x_test)),
    "number_of_classes": len(class_names),
}

metrics_path = REPORTS_DIR / "day6_fashion_mnist_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=4), encoding="utf-8")

print("Confusion matrix saved:", confusion_path)
print("Metrics saved:", metrics_path)
print(metrics)

## 12. Save the trained model

The model is saved in Keras `.keras` format. It can be loaded later without retraining.

In [ ]:
model_path = MODELS_DIR / "retailmind_fashion_mnist_cnn_day6.keras"
save_model(model, model_path)

print("Model saved:", model_path)

## 13. Predict the category of one test image

The prediction function accepts one preprocessed image and returns the predicted
category and its confidence score.

The confidence is the model's predicted probability for its chosen class; it is
not a guarantee that the prediction is correct.

In [ ]:
sample_index = 0
sample_image = x_test[sample_index]
actual_label = class_names[y_test[sample_index]]

predicted_label, confidence = predict_image(
    model=model,
    image=sample_image,
    class_names=class_names,
)

plt.imshow(sample_image.squeeze(), cmap="gray")
plt.axis("off")
plt.title(
    f"Actual: {actual_label}\nPredicted: {predicted_label} ({confidence:.2%})"
)
plt.show()

print("Actual category:", actual_label)
print("Predicted category:", predicted_label)
print(f"Confidence: {confidence:.2%}")

## 14. Load the saved model

This verifies that the model artifact can be loaded again from disk.

In [ ]:
loaded_model = tf.keras.models.load_model(model_path)
print("Model loaded successfully.")
loaded_model.summary()

## 15. Day 6 completion checklist

- [ ] Fashion-MNIST loaded directly from Keras.
- [ ] Training and test shapes verified.
- [ ] Pixel values normalized to 0–1.
- [ ] Sample images visualized with category names.
- [ ] CNN architecture built using `src/model.py`.
- [ ] Model trained using `src/train.py`.
- [ ] Test loss and accuracy calculated.
- [ ] Confusion matrix inspected and saved.
- [ ] Trained model saved in `models/`.
- [ ] Single-image prediction tested.
- [ ] Saved model loaded successfully.

## Files created after running

```text
RetailMind/
├── notebooks/
│   └── RetailMind_Day6_CNN_Fashion_MNIST.ipynb
├── src/
│   ├── __init__.py
│   ├── data_loader.py
│   ├── model.py
│   ├── train.py
│   ├── evaluate.py
│   └── predict.py
├── models/
│   └── retailmind_fashion_mnist_cnn_day6.keras
└── reports/
    ├── day6_fashion_mnist_confusion_matrix.csv
    └── day6_fashion_mnist_metrics.json
```

**Day 6 outcome:** RetailMind now has a reusable CNN image-classification module
that can classify the 10 Fashion-MNIST clothing categories.